# 🦥 SLOTH Cookbook
## Structural Loader with On-demand Traversal Handling

**Lazy by design. Fast by default.**

This comprehensive cookbook demonstrates how to use SLOTH for parsing, validating, modifying, and writing mmCIF files with elegant dot notation and high-performance gemmi backend.

## Table of Contents
1. Setup and Installation
2. Import Required Libraries
3. Understanding SLOTH's Core Components
4. Parsing mmCIF Files with Embedded Data
5. Exploring Data Structures with Dot Notation
6. Demonstrating 2D Slicing
7. Validating mmCIF Data
8. Modifying mmCIF Data
9. Creating Sample Data - Manual Approach
10. Creating Sample Data - Programmatic Approach
11. Creating Sample Data - Auto-Creation with Dot Notation
12. Exporting to Nested JSON
13. Importing from JSON
14. Round-Trip Validation
15. Writing Modified mmCIF Files
16. Complete Workflow Example

## 1. Setup and Installation

SLOTH can be installed via pip. Make sure you have Python 3.8 or higher.

In [1]:
# Install SLOTH (if not already installed)
# !pip install sloth-biosim

# Verify installation
import sloth
print(f"✅ SLOTH version: {sloth.__version__ if hasattr(sloth, '__version__') else 'installed'}")

✅ SLOTH version: 0.3.5


## 2. Import Required Libraries

Let's import all the SLOTH components we'll need for this cookbook.

In [2]:
import os
import json
import tempfile
from pathlib import Path

# SLOTH core components
from sloth.mmcif import (
    MMCIFHandler,
    ValidatorFactory,
    DataSourceFormat,
)

# SLOTH data models
from sloth.mmcif.models import MMCIFDataContainer, DataBlock, Category

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 3. Understanding SLOTH's Core Components

SLOTH provides an elegant, Pythonic API for working with mmCIF data:

- **MMCIFDataContainer**: The top-level container holding one or more data blocks
- **DataBlock**: A named collection of categories (like `data_1ABC`)
- **Category**: A collection of related items (like `_atom_site` or `_entity`)
- **Dot Notation**: Access data naturally like `container.data_1ABC._atom_site.Cartn_x`

### Key Features
- ✨ **Auto-creation**: Objects are created automatically as you access them
- 🚀 **High Performance**: Uses gemmi backend for fast parsing
- 🐍 **Pythonic**: Clean, intuitive API with dot notation
- 🔄 **Round-trip**: Full support for mmCIF → JSON → mmCIF conversions

## 4. Parsing mmCIF Files with Embedded Data

Let's parse a comprehensive protein-ligand complex structure. We'll use embedded demo data for convenience.

In [3]:
# Comprehensive demo mmCIF data with TRUE hierarchical relationships
# This structure demonstrates actual nesting with proper parent-child relationships:
#   entity -> entity_poly -> entity_poly_seq (3 levels)
#   entity -> struct_asym -> atom_site (3 levels)
COMPREHENSIVE_DEMO_MMCIF = """data_DEMO
#
# Entry information (top level)
#
_entry.id DEMO
#
# Database references
#
loop_
_database_2.database_id
_database_2.database_code
PDB DEMO
EMDB DEMO
#
# Entity information (parent level - root of hierarchy)
#
loop_
_entity.id
_entity.type
_entity.src_method
_entity.pdbx_description
1 polymer man 'Catalytic domain of model transferase'
2 water nat 'Water molecules'
#
# Entity polymer (child of entity via entity_id -> entity.id)
# Will nest under entity with id=1
#
loop_
_entity_poly.entity_id
_entity_poly.type
_entity_poly.nstd_chirality
_entity_poly.pdbx_seq_one_letter_code
1 'polypeptide(L)' no MAGLY
#
# Entity polymer sequence (child of entity_poly)
# Will nest under entity_poly, creating 3-level hierarchy
#
loop_
_entity_poly_seq.entity_id
_entity_poly_seq.num
_entity_poly_seq.mon_id
1 1 MET
1 2 ALA
1 3 GLY
1 4 LEU
1 5 TYR
#
# Structural asymmetric unit (child of entity)
# Creates parallel branch from entity
#
loop_
_struct_asym.id
_struct_asym.entity_id
_struct_asym.details
A 1 'Protein chain A'
W 2 'Water chain'
#
# Atom sites (child of struct_asym via label_asym_id -> struct_asym.id)
# Creates deeper nesting under struct_asym
#
loop_
_atom_site.group_PDB
_atom_site.id
_atom_site.type_symbol
_atom_site.label_atom_id
_atom_site.label_comp_id
_atom_site.label_asym_id
_atom_site.label_entity_id
_atom_site.label_seq_id
_atom_site.Cartn_x
_atom_site.Cartn_y
_atom_site.Cartn_z
_atom_site.occupancy
_atom_site.B_iso_or_equiv
_atom_site.pdbx_PDB_model_num
ATOM 1 N N MET A 1 1 20.154 6.718 46.973 1.00 25.00 1
ATOM 2 C CA MET A 1 1 21.618 6.765 47.254 1.00 24.50 1
ATOM 3 C C MET A 1 1 22.147 8.178 47.451 1.00 23.85 1
ATOM 4 N N ALA A 1 2 23.456 9.012 48.123 1.00 22.45 1
ATOM 5 C CA ALA A 1 2 24.123 10.234 48.567 1.00 21.30 1
HETATM 6 O O HOH W 2 . 12.345 15.678 35.432 1.00 18.56 1
HETATM 7 O O HOH W 2 . 13.456 16.789 36.543 1.00 19.67 1
#
"""

print("📝 Demo mmCIF data loaded with TRUE hierarchical relationships")
print("   🌲 3-level nesting: entity → entity_poly → entity_poly_seq")
print("   🌲 3-level nesting: entity → struct_asym → atom_site")
print("   ✨ This will create proper nested JSON output!")

📝 Demo mmCIF data loaded with TRUE hierarchical relationships
   🌲 3-level nesting: entity → entity_poly → entity_poly_seq
   🌲 3-level nesting: entity → struct_asym → atom_site
   ✨ This will create proper nested JSON output!


In [4]:
# Parse the embedded demo data
print("⚡ Parsing mmCIF data with gemmi backend...")

# Create a temporary file
with tempfile.NamedTemporaryFile(mode='w', suffix='.cif', delete=False) as tmp_file:
    tmp_file.write(COMPREHENSIVE_DEMO_MMCIF)
    tmp_filename = tmp_file.name

try:
    # Parse using MMCIFHandler
    handler = MMCIFHandler()
    mmcif = handler.read(tmp_filename)
    
    print(f"✅ Successfully parsed!")
    print(f"   Data blocks: {len(mmcif.data)}")
    
    if mmcif.data:
        block = mmcif.data[0]
        print(f"   Block name: '{block.name}'")
        print(f"   Categories: {len(block.categories)}")
        print(f"   Category names: {', '.join(block.categories)}")
finally:
    # Clean up
    if os.path.exists(tmp_filename):
        os.remove(tmp_filename)

⚡ Parsing mmCIF data with gemmi backend...
✅ Successfully parsed!
   Data blocks: 1
   Block name: 'DEMO'
   Categories: 7
   Category names: _entry, _database_2, _entity, _entity_poly, _entity_poly_seq, _struct_asym, _atom_site


## 5. Exploring Data Structures with Dot Notation

SLOTH's elegant dot notation makes accessing mmCIF data intuitive and Pythonic.

In [5]:
# Access data block using dot notation
block = mmcif.data_DEMO  # Elegant dot notation for accessing block by name!
print(f"🧱 Block name: {block.name}")

# Access categories using dot notation (elegant!)
if "_entry" in block.categories:
    entry_category = block._entry  # Dot notation in action!
    print(f"\n📂 Entry category:")
    print(f"   Entry ID: {entry_category.id[0]}")
    print(f"   Entry type: {entry_category.type[0] if hasattr(entry_category, 'type') else 'N/A'}")

# Access database information
if "_database_2" in block.categories:
    db_category = block._database_2  # Direct dot notation!
    print(f"\n💾 Database category:")
    print(f"   Database IDs: {db_category.database_id}")
    print(f"   Database codes: {db_category.database_code}")

# Access entity information
if "_entity" in block.categories:
    entity_category = block._entity  # Elegant dot notation!
    print(f"\n🧬 Entity category:")
    print(f"   Entity IDs: {entity_category.id}")
    print(f"   Entity types: {entity_category.type}")
    print(f"   Descriptions: {entity_category.pdbx_description}")

print("\n💡 Tip: Use mmcif.data_BLOCKNAME to access blocks and block._category_name.item_name for data!")

🧱 Block name: DEMO

📂 Entry category:
   Entry ID: DEMO
   Entry type: N/A

💾 Database category:
   Database IDs: <LazyGemmiColumn: 2 rows (not loaded)>
   Database codes: <LazyGemmiColumn: 2 rows (not loaded)>

🧬 Entity category:
   Entity IDs: <LazyGemmiColumn: 2 rows (not loaded)>
   Entity types: <LazyGemmiColumn: 2 rows (not loaded)>
   Descriptions: <LazyGemmiColumn: 2 rows (not loaded)>

💡 Tip: Use mmcif.data_BLOCKNAME to access blocks and block._category_name.item_name for data!


## 6. Demonstrating 2D Slicing

SLOTH supports both column-wise and row-wise access with powerful slicing capabilities.

In [6]:
# Column-wise access with dot notation
if "_atom_site" in block.categories:
    atom_site = block._atom_site
    
    print("📊 Column-wise access (the Pythonic way):")
    print(f"   Row count: {atom_site.row_count}")
    print(f"   Available items: {', '.join(atom_site.items[:5])}...")
    print(f"\n   Type symbols: {atom_site.type_symbol}")
    print(f"   X coordinates: {atom_site.Cartn_x}")
    print(f"   Y coordinates: {atom_site.Cartn_y}")
    print(f"   Z coordinates: {atom_site.Cartn_z}")
    
    # Row-wise access
    print(f"\n📋 Row-wise access (elegant and readable):")
    first_row = atom_site[0]
    print(f"   First atom:")
    print(f"     Type: {first_row.type_symbol}")
    print(f"     ID: {first_row.id}")
    print(f"     Position: ({first_row.Cartn_x}, {first_row.Cartn_y}, {first_row.Cartn_z})")
    
    # Slicing rows
    if atom_site.row_count >= 3:
        print(f"\n📑 Row slicing:")
        for i, row in enumerate(atom_site[0:3]):
            print(f"   Atom {i+1}: {row.type_symbol} at ({row.Cartn_x}, {row.Cartn_y}, {row.Cartn_z})")

print("\n💪 Dot notation makes your code readable, elegant, and Pythonic!")

📊 Column-wise access (the Pythonic way):
   Row count: 7
   Available items: group_PDB, id, type_symbol, label_atom_id, label_comp_id...

   Type symbols: <LazyGemmiColumn: 7 rows (not loaded)>
   X coordinates: <LazyGemmiColumn: 7 rows (not loaded)>
   Y coordinates: <LazyGemmiColumn: 7 rows (not loaded)>
   Z coordinates: <LazyGemmiColumn: 7 rows (not loaded)>

📋 Row-wise access (elegant and readable):
   First atom:
     Type: N
     ID: 1
     Position: (20.154, 6.718, 46.973)

📑 Row slicing:
   Atom 1: N at (20.154, 6.718, 46.973)
   Atom 2: C at (21.618, 6.765, 47.254)
   Atom 3: C at (22.147, 8.178, 47.451)

💪 Dot notation makes your code readable, elegant, and Pythonic!


## 7. Validating mmCIF Data

SLOTH provides a flexible validation framework for custom validators and cross-checkers.

In [7]:
# Define custom validator functions
def category_validator(category_name):
    """Example validator function."""
    print(f"  ✅ Validating category: {category_name}")
    return True

def cross_checker(category_name_1, category_name_2):
    """Example cross-checker function."""
    print(f"  🔗 Cross-checking: {category_name_1} ↔ {category_name_2}")
    return True

# Create validator factory and register validators
validator_factory = ValidatorFactory()

print("🛡️  Setting up validation...")
available_categories = block.categories[:2]

for cat_name in available_categories:
    validator_factory.register_validator(cat_name, category_validator)

# Register cross-checker if we have multiple categories
if len(available_categories) >= 2:
    cat_pair = (available_categories[0], available_categories[1])
    validator_factory.register_cross_checker(cat_pair, cross_checker)

# Run validation
print("\n🔍 Running validation...")
for cat_name in available_categories:
    validator_func = validator_factory.get_validator(cat_name)
    if validator_func:
        validator_func(cat_name)

# Run cross-validation
if len(available_categories) >= 2:
    cat1, cat2 = available_categories[0], available_categories[1]
    cross_checker_func = validator_factory.get_cross_checker((cat1, cat2))
    if cross_checker_func:
        cross_checker_func(cat1, cat2)

print("\n✅ Validation complete!")

🛡️  Setting up validation...

🔍 Running validation...
  ✅ Validating category: _entry
  ✅ Validating category: _database_2
  🔗 Cross-checking: _entry ↔ _database_2

✅ Validation complete!


## 8. Modifying mmCIF Data

Modify data elegantly using dot notation assignments.

In [8]:
print("✏️  Modifying data with dot notation...")

# Modify database information using elegant dot notation
if "_database_2" in block.categories:
    db_category = block._database_2
    
    print(f"\n📋 Original database_id: {db_category.database_id}")
    
    # Simple assignment with dot notation - change the last entry
    original_value = db_category.database_id[-1]
    db_category.database_id[-1] = "BMRB"  # Change EMDB to BMRB
    
    print(f"✏️  Modified database_id: '{original_value}' → '{db_category.database_id[-1]}'")
    print(f"   Using: block._database_2.database_id[-1] = 'BMRB'")
    print(f"\n📋 Updated database_id: {db_category.database_id}")

print("\n✅ Data modification complete!")

✏️  Modifying data with dot notation...

📋 Original database_id: <LazyGemmiColumn: 2 rows (not loaded)>
✏️  Modified database_id: 'EMDB' → 'BMRB'
   Using: block._database_2.database_id[-1] = 'BMRB'

📋 Updated database_id: LazyGemmiColumn(2 rows, loaded)

✅ Data modification complete!


## 9. Creating Sample Data - Manual Approach

The traditional approach: manually writing mmCIF format strings to files.

In [9]:
print("🖋️  Method 1: Manual mmCIF file creation")

# Create mmCIF content as a string
sample_content = """data_1ABC
_entry.id 1ABC_STRUCTURE
_database_2.database_id PDB
_database_2.database_code 1ABC
loop_
_atom_site.group_PDB
_atom_site.id
_atom_site.type_symbol
_atom_site.Cartn_x
_atom_site.Cartn_y
_atom_site.Cartn_z
ATOM 1 N 10.123 20.456 30.789
ATOM 2 C 11.234 21.567 31.890
"""

# Write to file
manual_file = "sample_manual.cif"
with open(manual_file, "w") as f:
    f.write(sample_content)

print(f"✅ Created manual sample: {manual_file}")

# Parse it back to verify
manual_mmcif = handler.read(manual_file)
print(f"✅ Verified: {len(manual_mmcif.data[0].categories)} categories")

🖋️  Method 1: Manual mmCIF file creation
✅ Created manual sample: sample_manual.cif
✅ Verified: 3 categories


## 10. Creating Sample Data - Programmatic Approach

Create mmCIF data programmatically using SLOTH's API with dictionary-style assignments.

In [10]:
print("⚙️  Method 2: Programmatic creation using dictionary notation")

# Create container and block
mmcif_prog = MMCIFDataContainer()
block_prog = DataBlock("1ABC")

# Create categories and add data
entry_category = Category("_entry")
entry_category["id"] = ["1ABC_STRUCTURE"]

database_category = Category("_database_2")
database_category["database_id"] = ["PDB"]
database_category["database_code"] = ["1ABC"]

atom_site_category = Category("_atom_site")
atom_site_category["group_PDB"] = ["ATOM", "ATOM"]
atom_site_category["id"] = ["1", "2"]
atom_site_category["type_symbol"] = ["N", "C"]
atom_site_category["Cartn_x"] = ["10.123", "11.234"]
atom_site_category["Cartn_y"] = ["20.456", "21.567"]
atom_site_category["Cartn_z"] = ["30.789", "31.890"]

# Add categories to block
block_prog["_entry"] = entry_category
block_prog["_database_2"] = database_category
block_prog["_atom_site"] = atom_site_category

# Add block to container
mmcif_prog["1ABC"] = block_prog

# Write using SLOTH handler
programmatic_file = "sample_programmatic.cif"
handler.write(mmcif_prog, programmatic_file)

print(f"✅ Created programmatic sample: {programmatic_file}")
print(f"✅ Categories: {len(mmcif_prog.data[0].categories)}")

⚙️  Method 2: Programmatic creation using dictionary notation
✅ Created programmatic sample: sample_programmatic.cif
✅ Categories: 3


## 11. Creating Sample Data - Auto-Creation with Dot Notation ✨

**This is SLOTH's most powerful feature!** Objects are automatically created as you access them using elegant dot notation.

In [11]:
print("✨ Method 3: Auto-creation with Elegant Dot Notation")
print("=" * 50)
print("SLOTH automatically creates nested objects!")
print()

# Create an empty container - this is all you need!
mmcif_auto = MMCIFDataContainer()

# Use dot notation to auto-create everything - just like magic!
mmcif_auto.data_1ABC._entry.id = ["1ABC_STRUCTURE"]
mmcif_auto.data_1ABC._database_2.database_id = ["PDB"]
mmcif_auto.data_1ABC._database_2.database_code = ["1ABC"]

# Add atom data
mmcif_auto.data_1ABC._atom_site.group_PDB = ["ATOM", "ATOM"]
mmcif_auto.data_1ABC._atom_site.id = ["1", "2"]
mmcif_auto.data_1ABC._atom_site.type_symbol = ["N", "C"]
mmcif_auto.data_1ABC._atom_site.Cartn_x = ["10.123", "11.234"]
mmcif_auto.data_1ABC._atom_site.Cartn_y = ["20.456", "21.567"]
mmcif_auto.data_1ABC._atom_site.Cartn_z = ["30.789", "31.890"]

# Write to file using handler
dot_notation_file = "sample_dot_notation.cif"
handler.write(mmcif_auto, dot_notation_file)

print(f"✅ Created dot notation sample: {dot_notation_file}")
print(f"\n🔍 What was auto-created:")
print(f"   📦 Container: {len(mmcif_auto)} block(s)")
print(f"   🧱 Block '1ABC': {len(mmcif_auto.data_1ABC.categories)} categories")
print(f"   📂 Categories: {', '.join(mmcif_auto.data_1ABC.categories)}")
print(f"\n💎 Elegant access examples:")
print(f"   Entry ID: {mmcif_auto.data_1ABC._entry.id[0]}")
print(f"   Database: {mmcif_auto.data_1ABC._database_2.database_id[0]}")
print(f"   Atom types: {mmcif_auto.data_1ABC._atom_site.type_symbol}")
print(f"\n🚀 Just write what you want, SLOTH creates what you need!")

✨ Method 3: Auto-creation with Elegant Dot Notation
SLOTH automatically creates nested objects!

✅ Created dot notation sample: sample_dot_notation.cif

🔍 What was auto-created:
   📦 Container: 1 block(s)
   🧱 Block '1ABC': 3 categories
   📂 Categories: _entry, _database_2, _atom_site

💎 Elegant access examples:
   Entry ID: 1ABC_STRUCTURE
   Database: PDB
   Atom types: ['N', 'C']

🚀 Just write what you want, SLOTH creates what you need!


## 12. Exporting to Nested JSON

SLOTH exports mmCIF data to nested JSON format, resolving parent-child relationships automatically. Categories like `entity_poly` are nested under their parent `entity`, and `entity_poly_seq` is nested under `entity_poly`.

In [12]:
print("📊 Demonstrating nested JSON export:")

# Create output directory
output_dir = "exports"
os.makedirs(output_dir, exist_ok=True)

# Export to JSON (nested structure with resolved relationships)
print("\n🔧 Exporting to Nested JSON:")
json_path = os.path.join(output_dir, "exported_nested.json")
handler.export(mmcif, file_path=json_path)

# Show file size
if os.path.exists(json_path):
    size = os.path.getsize(json_path)
    print(f"\n📁 Export Summary:")
    print(f"   File size: {size:,} bytes")
    
# Display the nested structure
print("\n🔍 Nested Structure Preview:")
with open(json_path, 'r') as f:
    data = json.load(f)
    
# Show the hierarchy
if 'data_DEMO' in data and '_entity' in data['data_DEMO']:
    entities = data['data_DEMO']['_entity']
    if entities:
        entity = entities[0]
        print(f"   📦 _entity (top level):")
        print(f"      - id: {entity.get('id')}")
        print(f"      - type: {entity.get('type')}")
        
        if '_entity_poly' in entity:
            print(f"      └─ 📦 _entity_poly (nested child):")
            poly = entity['_entity_poly'][0] if entity['_entity_poly'] else {}
            print(f"         - entity_id: {poly.get('entity_id')}")
            print(f"         - type: {poly.get('type')}")
            
            if '_entity_poly_seq' in poly:
                seq_list = poly['_entity_poly_seq']
                print(f"         └─ 📦 _entity_poly_seq (nested grandchild): {len(seq_list)} residues")
                if seq_list:
                    print(f"            - First residue: {seq_list[0].get('mon_id')} at position {seq_list[0].get('num')}")

print("\n✨ All category names have the '_' prefix, whether nested or not!")

📊 Demonstrating nested JSON export:

🔧 Exporting to Nested JSON:
📦 Using cached mapping rules
📦 Using cached dictionary data
Exported nested JSON to: exports/exported_nested.json

📁 Export Summary:
   File size: 5,799 bytes

🔍 Nested Structure Preview:
   📦 _entity (top level):
      - id: 1
      - type: polymer
      └─ 📦 _entity_poly (nested child):
         - entity_id: 1
         - type: 'polypeptide(L)'
         └─ 📦 _entity_poly_seq (nested grandchild): 5 residues
            - First residue: MET at position 1

✨ All category names have the '_' prefix, whether nested or not!


## 13. Importing from JSON

Import previously exported JSON files back into SLOTH's data structures.

In [13]:
print("📥 Demonstrating import functionality:")

# Import from nested JSON
print("\n✅ Importing from JSON:")
if os.path.exists(json_path):
    imported_container = handler.load(json_path)
    print(f"   ✅ Successfully imported: {json_path}")
    print(f"      Data blocks: {len(imported_container.data)}")
    if imported_container.data:
        imported_block = imported_container.data[0]
        print(f"      Categories: {len(imported_block.categories)}")
        print(f"      Category names: {', '.join(imported_block.categories[:5])}...")
else:
    print(f"   ❌ File not found: {json_path}")

📥 Demonstrating import functionality:

✅ Importing from JSON:
   ✅ Successfully imported: exports/exported_nested.json
      Data blocks: 1
      Categories: 7
      Category names: _entry, _database_2, _entity, _entity_poly, _entity_poly_seq...


## 14. Round-Trip Validation

Validate data integrity by comparing original and imported data.

In [14]:
print("🔄 Demonstrating round-trip validation:")

# Compare original and imported data
original_block = mmcif.data[0]
imported_block = imported_container.data[0]

print(f"\n📊 Comparing original vs imported:")
print(f"   Original categories: {len(original_block.categories)}")
print(f"   Imported categories: {len(imported_block.categories)}")

# Find common categories
common_categories = set(original_block.categories).intersection(
    set(imported_block.categories)
)
print(f"   ✓ Common categories: {len(common_categories)}")

# Check a sample category in detail
if common_categories:
    sample_cat = sorted(common_categories)[0]  # Sort for deterministic output
    print(f"\n🔍 Checking category: {sample_cat}")
    
    original_cat = original_block[sample_cat]
    imported_cat = imported_block[sample_cat]
    
    # Compare item names
    original_items = set(original_cat.items)
    imported_items = set(imported_cat.items)
    common_items = original_items.intersection(imported_items)
    
    if common_items:
        sample_item = sorted(common_items)[0]  # Sort for deterministic output
        original_values = original_cat[sample_item]
        imported_values = imported_cat[sample_item]
        
        print(f"   Item: '{sample_item}'")
        print(f"   Original values: {len(original_values)} items")
        print(f"   Imported values: {len(imported_values)} items")
        
        if len(original_values) > 0 and len(imported_values) > 0:
            if original_values[0] == imported_values[0]:
                print(f"   ✅ First value matches: '{original_values[0]}'")
            else:
                print(f"   ⚠️ First value differs!")

print("\n✅ Round-trip validation complete!")

🔄 Demonstrating round-trip validation:

📊 Comparing original vs imported:
   Original categories: 7
   Imported categories: 7
   ✓ Common categories: 7

🔍 Checking category: _atom_site
   Item: 'B_iso_or_equiv'
   Original values: 7 items
   Imported values: 7 items
   ✅ First value matches: '25.00'

✅ Round-trip validation complete!


## 15. Writing Modified mmCIF Files

Write modified mmCIF data to files using the handler's write method.

In [15]:
print("💾 Writing modified mmCIF file:")

# Write the modified container to a new file
output_file = "demo_modified.cif"
handler.write(mmcif, output_file)

print(f"✅ Written to: {output_file}")

# Verify the output
print("\n🔍 Verifying output:")
verify_data = handler.read(output_file)
verify_block = verify_data.data[0]

print(f"   ✅ Data blocks: {len(verify_data.data)}")
print(f"   ✅ Categories: {len(verify_block.categories)}")
print(f"   ✅ Block name: '{verify_block.name}'")

# Show a sample of the data
if "_database_2" in verify_block.categories:
    db_cat = verify_block._database_2
    print(f"\n📋 Verification - Database information:")
    print(f"   Database IDs: {db_cat.database_id}")
    print(f"   Database codes: {db_cat.database_code}")

💾 Writing modified mmCIF file:
✅ Written to: demo_modified.cif

🔍 Verifying output:
   ✅ Data blocks: 1
   ✅ Categories: 7
   ✅ Block name: 'DEMO'

📋 Verification - Database information:
   Database IDs: <LazyGemmiColumn: 2 rows (not loaded)>
   Database codes: <LazyGemmiColumn: 2 rows (not loaded)>


## 16. Complete Workflow Example

Let's put everything together in a complete workflow!

In [16]:
print("🚀 Complete SLOTH Workflow")
print("=" * 50)

# Step 1: Create a new container with auto-creation
print("\n1️⃣ Creating new mmCIF data with dot notation...")
workflow_mmcif = MMCIFDataContainer()

# Auto-create structure with dot notation
workflow_mmcif.data_WORKFLOW._entry.id = ["WORKFLOW_DEMO"]
workflow_mmcif.data_WORKFLOW._database_2.database_id = ["PDB", "BMRB"]
workflow_mmcif.data_WORKFLOW._database_2.database_code = ["WORK", "WORK"]
workflow_mmcif.data_WORKFLOW._entity.id = ["1", "2"]
workflow_mmcif.data_WORKFLOW._entity.type = ["polymer", "non-polymer"]
workflow_mmcif.data_WORKFLOW._entity.pdbx_description = ["Protein", "Ligand"]

print("   ✅ Created with auto-creation!")

# Step 2: Validate the data
print("\n2️⃣ Validating data...")
validator = ValidatorFactory()
validator.register_validator("_entry", category_validator)
validator.get_validator("_entry")("_entry")
print("   ✅ Validation passed!")

# Step 3: Modify the data
print("\n3️⃣ Modifying data...")
workflow_mmcif.data_WORKFLOW._entry.id[0] = "MODIFIED_WORKFLOW"
print(f"   ✅ Modified entry ID to: {workflow_mmcif.data_WORKFLOW._entry.id[0]}")

# Step 4: Export to JSON
print("\n4️⃣ Exporting to JSON...")
workflow_json = os.path.join(output_dir, "workflow.json")
handler.export(workflow_mmcif, file_path=workflow_json)
print(f"   ✅ Exported to: {workflow_json}")

# Step 5: Import from JSON
print("\n5️⃣ Importing from JSON...")
reimported = handler.load(workflow_json)
print(f"   ✅ Reimported! Categories: {len(reimported.data[0].categories)}")

# Step 6: Write to mmCIF file
print("\n6️⃣ Writing final mmCIF file...")
final_file = "workflow_complete.cif"
handler.write(reimported, final_file)
print(f"   ✅ Written to: {final_file}")

# Step 7: Verify round-trip
print("\n7️⃣ Verifying round-trip integrity...")
original_cats = len(workflow_mmcif.data[0].categories)
final_cats = len(reimported.data[0].categories)
print(f"   Original categories: {original_cats}")
print(f"   Final categories: {final_cats}")
print(f"   ✅ Round-trip successful!" if original_cats == final_cats else "   ⚠️ Category count changed")

print("\n" + "=" * 50)
print("🎉 Complete workflow finished successfully!")
print("💡 SLOTH makes mmCIF manipulation elegant and powerful!")

🚀 Complete SLOTH Workflow

1️⃣ Creating new mmCIF data with dot notation...
   ✅ Created with auto-creation!

2️⃣ Validating data...
  ✅ Validating category: _entry
   ✅ Validation passed!

3️⃣ Modifying data...
   ✅ Modified entry ID to: MODIFIED_WORKFLOW

4️⃣ Exporting to JSON...
📦 Using cached mapping rules
📦 Using cached dictionary data
Exported nested JSON to: exports/workflow.json
   ✅ Exported to: exports/workflow.json

5️⃣ Importing from JSON...
   ✅ Reimported! Categories: 3

6️⃣ Writing final mmCIF file...
   ✅ Written to: workflow_complete.cif

7️⃣ Verifying round-trip integrity...
   Original categories: 3
   Final categories: 3
   ✅ Round-trip successful!

🎉 Complete workflow finished successfully!
💡 SLOTH makes mmCIF manipulation elegant and powerful!


## Summary

You've learned how to:

✅ **Parse** mmCIF files with high-performance gemmi backend  
✅ **Access** data elegantly using dot notation  
✅ **Slice** data both column-wise and row-wise  
✅ **Validate** data with custom validators  
✅ **Modify** data with simple assignments  
✅ **Create** data three ways: manual, programmatic, and auto-creation  
✅ **Export** to nested JSON with automatic relationship resolution  
✅ **Import** from JSON with full round-trip support  
✅ **Write** modified mmCIF files  
✅ **Execute** complete workflows combining all features

### Key Takeaways

🦥 **Lazy by design, fast by default** - SLOTH combines elegant APIs with high performance  
✨ **Auto-creation** - Objects are created automatically as you access them  
🐍 **Pythonic** - Dot notation makes code readable and intuitive  
🔄 **Round-trip support** - Full mmCIF → JSON → mmCIF conversion  
🌲 **Nested JSON** - Automatically resolves parent-child relationships for hierarchical data

### Next Steps

- Explore the [SLOTH documentation](https://github.com/lucas-ebi/sloth)
- Check out real-world examples in the repository
- Contribute to the project on GitHub

**Happy coding with SLOTH!** 🦥